In [3]:
import pandas as pd
import numpy as np
from datetime import timedelta

# Cố định seed
np.random.seed(42)

# ==========================================
# 1. ĐỌC DỮ LIỆU ĐẦU VÀO TỪ FILE OLIST THỰC TẾ
# ==========================================
# Đọc file csv hiển thị trong ảnh
df_orders_full = pd.read_csv('olist_orders_dataset.csv')

# Ép kiểu dữ liệu thời gian cho 2 cột quan trọng
df_orders_full['order_estimated_delivery_date'] = pd.to_datetime(df_orders_full['order_estimated_delivery_date'])
df_orders_full['order_delivered_customer_date'] = pd.to_datetime(df_orders_full['order_delivered_customer_date'])

# Lấy thử 100 đơn hàng đầu tiên để chạy prototype cho nhanh
df_orders = df_orders_full.head(100).copy()

# Tính toán SLA Delay (Tín hiệu chính)
df_orders['sla_delay_days'] = (df_orders['order_delivered_customer_date'] - df_orders['order_estimated_delivery_date']).dt.days
df_orders['sla_delay_days'] = df_orders['sla_delay_days'].apply(lambda x: x if x > 0 else 0).fillna(0)


# ==========================================
# 2. HÀM MÔ PHỎNG MONTE CARLO & BACKCASTING
# ==========================================
def simulate_delivery_attempts(order_row):
    attempts = []
    order_id = order_row['order_id']
    status = order_row['order_status']
    
    # ---------------------------------------
    # Trường hợp 1: Đơn bị hủy (Canceled) / Không giao thành công
    # ---------------------------------------
    if status != 'delivered' or pd.isna(order_row['order_delivered_customer_date']):
        # Ép kiểu int() để sửa lỗi numpy.int32
        num_fails = int(np.random.choice([1, 2], p=[0.7, 0.3]))
        base_date = order_row['order_estimated_delivery_date'] 
        
        if pd.isna(base_date):
            return attempts 
            
        for i in range(num_fails):
            # Lỗi đã được xử lý bằng cách ép num_fails về int ở trên
            attempt_date = base_date - timedelta(days=(num_fails - 1 - i))
            attempts.append({
                'order_id': order_id,
                'attempt_number': i + 1,
                'attempt_date_key': attempt_date.date(),
                'attempt_outcome': 'failed',
                'sla_risk_score': 0.9, 
                'is_final_attempt_flag': True if i == (num_fails - 1) else False
            })
        return attempts

    # ---------------------------------------
    # Trường hợp 2: Đơn Delivered (Sự thật tuyệt đối)
    # ---------------------------------------
    delivered_date = order_row['order_delivered_customer_date']
    sla_delay = order_row['sla_delay_days']
    
    zone_risk = np.random.uniform(0.0, 0.3)
    weather_risk = np.random.uniform(0.0, 0.4)
    holiday_risk = np.random.choice([0.0, 0.3], p=[0.9, 0.1])
    
    base_fail_prob = 0.05 + (sla_delay * 0.15) + zone_risk + weather_risk + holiday_risk
    base_fail_prob = min(base_fail_prob, 0.85) 
    
    simulated_attempt_dates = [delivered_date] 
    current_eval_date = delivered_date
    
    max_lookback = int(sla_delay) + 2 
    
    for _ in range(max_lookback):
        roll = np.random.uniform(0, 1)
        
        if roll < base_fail_prob:
            # Dùng int() đảm bảo không sinh lỗi timedelta tại đây
            days_to_subtract = int(np.random.randint(1, 3))
            current_eval_date -= timedelta(days=days_to_subtract)
            simulated_attempt_dates.insert(0, current_eval_date)
            base_fail_prob *= 0.6 
        else:
            break 
            
    total_attempts = len(simulated_attempt_dates)
    for idx, date in enumerate(simulated_attempt_dates):
        is_final = (idx == total_attempts - 1)
        
        sla_risk_score = round(min(1.0, zone_risk + weather_risk + holiday_risk + (0.1 if not is_final else 0)), 2)
        
        attempts.append({
            'order_id': order_id,
            'attempt_number': idx + 1,
            'attempt_date_key': date.date(),
            'attempt_outcome': 'delivered' if is_final else 'failed',
            'sla_risk_score': sla_risk_score,
            'is_final_attempt_flag': is_final
        })
        
    return attempts

# ==========================================
# 3. THỰC THI & KIỂM TRA
# ==========================================
all_attempts = []
for index, row in df_orders.iterrows():
    all_attempts.extend(simulate_delivery_attempts(row))

df_fact_delivery_attempts = pd.DataFrame(all_attempts)
display(df_fact_delivery_attempts)

# Bạn có thể lọc thử một mã order_id bất kỳ để xem chuỗi attempt của nó:
# display(df_fact_delivery_attempts[df_fact_delivery_attempts['attempt_number'] > 1])

,order_id,attempt_number,attempt_date_key,attempt_outcome,sla_risk_score,is_final_attempt_flag
0,e481f51cbdc54678b7cc49136f2d6af7,1,2017-10-10,delivered,0.49,True
1,53cdb2fc8bc7dce0b6741e2150273451,1,2018-08-07,delivered,0.11,True
2,47770eb9100c2d0c44946d9cf07ec65d,1,2018-08-17,delivered,0.46,True
3,949d5b44dbf5de918fe9c16f97b45f8a,1,2017-11-30,failed,0.43,False
4,949d5b44dbf5de918fe9c16f97b45f8a,2,2017-12-02,delivered,0.33,True
...,...,...,...,...,...,...
149,6a0a8bfbbe700284feb0845d95e0867f,2,2017-12-27,failed,0.70,False
150,6a0a8bfbbe700284feb0845d95e0867f,3,2017-12-28,delivered,0.60,True
151,f7959f8385f34c4f645327465a1c9fc4,1,2017-04-08,failed,0.42,False
152,f7959f8385f34c4f645327465a1c9fc4,2,2017-04-10,delivered,0.32,True


In [5]:
display(df_fact_delivery_attempts[df_fact_delivery_attempts['attempt_number'] > 1])

,order_id,attempt_number,attempt_date_key,attempt_outcome,sla_risk_score,is_final_attempt_flag
4,949d5b44dbf5de918fe9c16f97b45f8a,2,2017-12-02,delivered,0.33,True
7,a4591c265e18cb1dcee52889e2d8acc3,2,2017-07-24,failed,0.50,False
8,a4591c265e18cb1dcee52889e2d8acc3,3,2017-07-26,delivered,0.40,True
10,136cce7faa42fdb2cefd53fdc79a6098,2,2017-05-09,failed,0.90,True
12,6514b8ad8028c9f2cc2374ded245783f,2,2017-05-26,delivered,0.48,True
14,76c6e866289321a7c93b82b54852dc33,2,2017-01-31,failed,0.71,False
15,76c6e866289321a7c93b82b54852dc33,3,2017-02-02,delivered,0.61,True
24,403b97836b0c04a622354cf531062e5f,2,2018-01-18,failed,0.52,False
25,403b97836b0c04a622354cf531062e5f,3,2018-01-20,delivered,0.42,True
27,116f0b09343b49556bbad5f35bee0cdf,2,2018-01-08,delivered,0.50,True
